In [35]:
from sklearn.base import clone
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# Model eval

### functions

In [ ]:
def get_models(random_state=42):
    """Returns a dictionary of machine learning models with their corresponding pipelines."""

    return {
        "SVM-RBF": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),

            ("clf", SVC(
                kernel="rbf",
                C=1.0,
                gamma="scale",
                class_weight="balanced",
                probability=True,
                random_state=random_state
            ))
        ]),
        "SVM-Linear": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="linear",
                C=1.0,
                gamma="scale",
                class_weight="balanced",
                probability=True,
                random_state=random_state
            ))
        ]),
        "KNN": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(n_neighbors=5, weights="distance"))
        ]),

        "RandomForest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", RandomForestClassifier(
                n_estimators=500,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1
            ))
        ]),

        "XGBoost": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", XGBClassifier(
                n_estimators=300,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=random_state,
            n_jobs=-1
            ))
        ]),
        "MLP": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation="relu",
                alpha=1e-3,
                learning_rate_init=1e-3,
                max_iter=1000,
                early_stopping=True,
                random_state=random_state
            ))
        ]
    )
    }

def compute_metrics_from_predictions(predictions_df, on_columns="language", filter_values=None):
    """
    Computes classification metrics from a DataFrame containing predictions.
    predictions_df (pd.DataFrame):DataFrame containing columns 'model','feature_set', 'y_true', 'y_pred', 'prob_class_0', and 'prob_class_1'.
    on_columns (str or list): Column(s) to filter the DataFrame on.
    filter_values (list): Values to filter the specified columns on.
    Returns:
        metrics_df (pd.DataFrame): DataFrame containing overall metrics for each model and feature set.
        per_class_metrics_df (pd.DataFrame): DataFrame containing per-class metrics for each model and feature set.
    """
    metrics_rows = []
    per_class_rows = []
    # for col, values in zip(on_columns, filter_values):
    #     if col not in predictions_df.columns:
    #         raise ValueError(f"Column '{col}' not found in predictions DataFrame.")
    #     predictions_df = predictions_df[predictions_df[col].isin(values)]
    if filter_values is not None:
        predictions_df = predictions_df[predictions_df[on_columns].isin(filter_values)]
    group_cols = ["model", "feature_set"]

    for (model_name, feature_set), df_group in predictions_df.groupby(group_cols):
        y_true = df_group["y_true"].values
        y_pred = df_group["y_pred"].values
        prob_class_0 = df_group["prob_class_0"].values
        prob_class_1 = df_group["prob_class_1"].values
        classes = np.sort(np.unique(y_true))

        row = {
            "model": model_name,
            "feature_set": feature_set,
            "accuracy": accuracy_score(y_true, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "auc": roc_auc_score(y_true, prob_class_1) if len(classes) == 2 else None
        }

        metrics_rows.append(row)

        report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
        report_df = pd.DataFrame(report).T.reset_index().rename(columns={"index": "class_or_avg"})
        report_df["model"] = model_name
        report_df["feature_set"] = feature_set
        per_class_rows.append(report_df)

    metrics_df = pd.DataFrame(metrics_rows)
    per_class_metrics_df = pd.concat(per_class_rows, ignore_index=True)

    return metrics_df, per_class_metrics_df

In [ ]:
# splits
def load_split_df(csv_path):
    """Load the split DataFrame from a CSV file and preprocess it."""
    split_df = pd.read_csv(csv_path).copy()
    split_df["split"] = split_df["split"].astype(str).str.strip().str.lower()
    return split_df

def make_split_based_subset(
    df,
    csv_path,
    languages,
    split_column="split",
    train_value="train",
    test_value="test",
):
    """Create train and test subsets based on a split CSV file and specified languages. """
    split_df = load_split_df(csv_path)
    merged_df = pd.merge(df, split_df[['file_id','label', 'split']], on=("file_id",'label'), how="left")
    key_columns = ["file_id", "label"]

    missing_in_features = [c for c in key_columns if c not in df.columns]
    missing_in_split = [c for c in key_columns if c not in split_df.columns]

    if missing_in_features:
        raise ValueError(f"features_df missing key columns: {missing_in_features}")
    if missing_in_split:
        raise ValueError(f"split_df missing key columns: {missing_in_split}")
    if "split" not in split_df.columns:
        raise ValueError("split_df must contain a 'split' column")
    

    subset_df = merged_df[merged_df["language"].isin(languages)].copy()

    if subset_df.empty:
        raise ValueError(
            f"No rows found for languages={languages}. "
            f"Available languages: {sorted(merged_df['language'].dropna().astype(str).unique().tolist())}"
        )

    split_series = subset_df[split_column].astype(str).str.strip().str.lower()

    train_df = subset_df[split_series == str(train_value).lower()].copy()
    test_df = subset_df[split_series == str(test_value).lower()].copy()

    if train_df.empty or test_df.empty:
        raise ValueError(
            f"Split-based subset produced empty train/test. "
            f"Train rows={len(train_df)}, test rows={len(test_df)}, languages={languages}"
        )

    return train_df, test_df

# def make_cross_language_split(features_df, train_languages, test_languages):
#     train_df = features_df[features_df["language"].isin(train_languages)].copy()
#     test_df = features_df[features_df["language"].isin(test_languages)].copy()

#     if train_df.empty:
#         raise ValueError(f"No train rows found for train_languages={train_languages}")
#     if test_df.empty:
#         raise ValueError(f"No test rows found for test_languages={test_languages}")

#     return train_df, test_df
def make_cross_language_split(
    features_df,
    train_languages,
    test_languages,
    csv_path,
    train_value="train",
    test_value="test",
):
    """Create train and test subsets based on a split CSV file and specified languages for cross-lingual evaluation."""
    split_df = load_split_df(csv_path)

    merged_df = pd.merge(
        features_df,
        split_df[["file_id", "label", "split"]],
        on=["file_id", "label"],
        how="left",
    )

    split_series = (
        merged_df["split"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Only training-set samples from the two source languages
    train_df = merged_df[
        merged_df["language"].isin(train_languages)
        & (split_series == train_value.lower())
    ].copy()

    # Only test-set samples from the held-out language
    test_df = merged_df[
        merged_df["language"].isin(test_languages)
        & (split_series == test_value.lower())
    ].copy()

    if train_df.empty:
        raise ValueError(
            f"No training rows found for languages={train_languages}"
        )

    if test_df.empty:
        raise ValueError(
            f"No test rows found for languages={test_languages}"
        )

    print(
        f"Cross-lingual split: "
        f"{len(train_df)} training samples from {train_languages}; "
        f"{len(test_df)} test samples from {test_languages}"
    )

    return train_df, test_df

In [ ]:
# table manager
def load_feature_tables(feature_paths_by_name):
    """
    feature_paths_by_name: dict like
    {
        "en": "data/processed/features_en.csv",
        "es": "data/processed/features_es.csv",
        "zh": "data/processed/features_zh.csv",
    }
    """
    tables = {}

    for name, path in feature_paths_by_name.items():
        df = pd.read_csv(path)
        tables[name] = df

    return tables

def subset_by_languages(df, languages):
    """
    getting the subset of the dataframe based on the specified languages.
    df: dataframe with a "language" column
    languages: list of languages to keep
    """
    return df[df["language"].isin(languages)].copy()

def build_master_feature_table(feature_tables):
    """
    feature_tables: dict of name -> df
    Returns a concatenated master dataframe.
    """
    df = pd.concat(list(feature_tables.values()), ignore_index=True)
    return df

In [ ]:
def train_and_predict(
    train_df,
    test_df,
    feature_columns,
    models,
    experiment_name,
    strategy,
    feature_set_name,
    train_languages,
    test_languages,
    target_column="label",
    sample_id_column="file_id",
    language_column="language",
    dataset_column="dataset",
):
    """
    Train models on the training set and predict on the test set.
    train_df: DataFrame containing training data
    test_df: DataFrame containing test data
    feature_columns: List of feature column names to use for training
    models: Dictionary of model names and their corresponding sklearn model instances
    experiment_name: Name of the experiment
    strategy: Strategy used for the experiment 
    feature_set_name: Name of the feature set used
    train_languages: List of languages used for training
    test_languages: List of languages used for testing
    target_column: Name of the target column 
    sample_id_column: Name of the sample ID column 
    language_column: Name of the language column 
    dataset_column: Name of the dataset column 
    Returns a DataFrame containing predictions and related information.
    """
    # getting x train and y train from the df
    X_train = train_df[feature_columns].copy()
    y_train = train_df[target_column].values

    # getting x test and y test from the df
    X_test = test_df[feature_columns].copy()
    y_test = test_df[target_column].values
    prediction_rows = []

    # loop through each model, train it, and make predictions
    for model_name, model in models.items():
        clf = clone(model)
        clf.fit(X_train, y_train)

        y_pred = clf.predict(X_test)

        row_dict = {
            "sample_id": test_df[sample_id_column].values,
            "language": test_df[language_column].values,
            "dataset": test_df[dataset_column].values if dataset_column in test_df.columns else ["unknown"] * len(test_df),
            "model": model_name,
            "feature_set": [feature_set_name] * len(test_df),
            "experiment_name": [experiment_name] * len(test_df),
            "strategy": [strategy] * len(test_df),
            "train_languages": [",".join(train_languages)] * len(test_df),
            "test_languages": [",".join(test_languages)] * len(test_df),
            "y_true": y_test,
            "y_pred": y_pred,
        }

        if hasattr(clf, "predict_proba"):
            y_proba = clf.predict_proba(X_test)
            for class_idx in range(y_proba.shape[1]):
                row_dict[f"prob_class_{class_idx}"] = y_proba[:, class_idx]

        prediction_rows.append(pd.DataFrame(row_dict))

    return pd.concat(prediction_rows, ignore_index=True)

In [ ]:
def run_experiment(
    features_df,
    feature_columns,
    models,
    experiment,
    feature_set_name,
    target_column="label",
):
    """
    Run a single experiment based on the provided configuration.
    features_df: DataFrame containing the features and metadata
    feature_columns: List of feature column names to use for training
    models: Dictionary of model names and their corresponding sklearn model instances
    experiment: Dictionary containing experiment configuration (name, strategy, languages, etc.)
    feature_set_name: Name of the feature set used
    target_column: Name of the target column (default: "label")
    """
    strategy = experiment["strategy"]
    print(f"Running experiment: {experiment['name']} with strategy: {strategy}")
    if strategy == "mono":
        language = experiment["language"]
        print(f"Running mono-language experiment for language: {language}")

        train_df, test_df = make_split_based_subset(
            df=features_df,
            csv_path=experiment["csv_path"],
            languages=language
        )

        print(f"Train set size: {len(train_df)}, Test set size: {len(test_df)}")

        return train_and_predict(
            train_df=train_df,
            test_df=test_df,
            feature_columns=feature_columns,
            models=models,
            experiment_name=experiment["name"],
            strategy=strategy,
            feature_set_name=feature_set_name,
            train_languages=language,
            test_languages=language,
            target_column=target_column,
        )

    elif strategy == "cross":

        train_df, test_df = make_cross_language_split(
        features_df,
        csv_path=experiment["csv_path"],
        train_languages=experiment["train_languages"],
        test_languages=experiment["test_languages"],
        )

        return train_and_predict(
            train_df=train_df,
            test_df=test_df,
            feature_columns=feature_columns,
            models=models,
            experiment_name=experiment["name"],
            strategy=strategy,
            feature_set_name=feature_set_name,
            train_languages=experiment["train_languages"],
            test_languages=experiment["test_languages"],
            target_column=target_column,
        )

    elif strategy == "multi":
        language = experiment["language"]
        train_df, test_df = make_split_based_subset(
            df=features_df,
            languages=language,
            csv_path=experiment["csv_path"],
        )

        return train_and_predict(
            train_df=train_df,
            test_df=test_df,
            feature_columns=feature_columns,
            models=models,
            experiment_name=experiment["name"],
            strategy=strategy,
            feature_set_name=feature_set_name,
            train_languages=language,
            test_languages=language,
            target_column=target_column,
        )
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

In [49]:
def run_all_experiments(
    features_df,
    feature_sets,
    experiments,
    get_models_fn,
    random_state=42,
):
    all_predictions = []

    for feature_set_name, feature_columns in feature_sets.items():
        print(f"\nRunning feature set: {feature_set_name}")
        models = get_models_fn(random_state=random_state)

        for experiment in experiments:
            print(f"  Experiment: {experiment['name']} [{experiment['strategy']}]")

            preds = run_experiment(
                features_df=features_df,
                feature_columns=feature_columns,
                models=models,
                experiment=experiment,
                feature_set_name=feature_set_name,
                target_column="label",
            )

            all_predictions.append(preds)

    return pd.concat(all_predictions, ignore_index=True)

### running

In [ ]:
# configuration of all the experiments
EXPERIMENTS = [
        {
        "name": "mono_english",
        "strategy": "mono",
        "language": ["English"],
        "csv_path": r"D:\masteruwefduyqeahfdqe\ASR-project\FIN_split_70_30_by_language_grouped_by_safe_speaker.csv",
    },
    {
        "name": "mono_mandarin",
        "strategy": "mono",
        "language": ["Mandarin"],
        "csv_path": r"D:\masteruwefduyqeahfdqe\ASR-project\FIN_split_70_30_by_language_grouped_by_safe_speaker.csv",
    },
    {
        "name": "mono_greek",
        "strategy": "mono",
        "language": ["Greek"],
        "csv_path": r"D:\masteruwefduyqeahfdqe\ASR-project\FIN_split_70_30_by_language_grouped_by_safe_speaker.csv",

    },
    {
        "name": "train_english_greek_test_mandarin",
        "strategy": "cross",
        "train_languages": ["English", "Greek"],
        "test_languages": ["Mandarin"],
        "csv_path": r"D:\masteruwefduyqeahfdqe\ASR-project\FIN_split_70_30_by_language_grouped_by_safe_speaker.csv",
    },
        {
        "name": "train_mandarin_greek_test_english",
        "strategy": "cross",
        "train_languages": ["Mandarin", "Greek"],
        "test_languages": ["English"],
        "csv_path": r"D:\masteruwefduyqeahfdqe\ASR-project\FIN_split_70_30_by_language_grouped_by_safe_speaker.csv",
    },
        {
        "name": "train_english_mandarin_test_greek",
        "strategy": "cross",
        "train_languages": ["English", "Mandarin"],
        "test_languages": ["Greek"],
        "csv_path": r"D:\masteruwefduyqeahfdqe\ASR-project\FIN_split_70_30_by_language_grouped_by_safe_speaker.csv",
    },
    {
        "name": "train_mandarin_english_greek",
        "strategy": "multi",
        "language": ["English", "Mandarin", "Greek"],
        "csv_path": r"D:\masteruwefduyqeahfdqe\ASR-project\FIN_split_70_30_by_language_grouped_by_safe_speaker.csv",    
    }
]

In [51]:
FEATURE_SETS = {
    "all": [
        # 'word_count',
        "total_speaking_time_normalized",
        "interviewer_interruptions",
        "word_rate_words_per_sec",
        # "word_rate_overall",
        "mean_utterance_word_rate_words_per_sec",
        "median_utterance_word_rate_words_per_sec",
        "std_utterance_word_rate_words_per_sec",
        "unique_word_count",
        "type_token_ratio",
        "mean_words_per_utterance",
        "median_words_per_utterance",
        'mean_unique_words_per_utterance',
        'median_unique_words_per_utterance',
        'std_unique_words_per_utterance',
        "std_words_per_utterance",
        "mean_utterance_duration_sec",
        "median_utterance_duration_sec",
        "std_utterance_duration_sec",
        "MeanUnvoicedSegmentLength",
        "StddevUnvoicedSegmentLength",
        "F0semitoneFrom27.5Hz_sma3nz_amean",
        "F0semitoneFrom27.5Hz_sma3nz_stddevNorm",
        "F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2",
        "loudness_sma3_amean",
        "loudness_sma3_stddevNorm",
        "loudness_sma3_pctlrange0-2",
        "loudnessPeaksPerSec",
        "jitterLocal_sma3nz_amean",
        "shimmerLocaldB_sma3nz_amean",
        "HNRdBACF_sma3nz_amean",
        "alphaRatioV_sma3nz_amean",
        "slopeV0-500_sma3nz_amean",
        "slopeV500-1500_sma3nz_amean",
        "equivalentSoundLevel_dBp",
    ],
    "cha_only": [
        "total_speaking_time_normalized",
        "interviewer_interruptions",
        "word_rate_words_per_sec",
        # "word_rate_overall",
        "mean_utterance_word_rate_words_per_sec",
        "median_utterance_word_rate_words_per_sec",
        "std_utterance_word_rate_words_per_sec",
        "unique_word_count",
        "type_token_ratio",
        "mean_words_per_utterance",
        "median_words_per_utterance",
        'mean_unique_words_per_utterance',
        'median_unique_words_per_utterance',
        'std_unique_words_per_utterance',
        "std_words_per_utterance",
        "mean_utterance_duration_sec",
        "median_utterance_duration_sec",
        "std_utterance_duration_sec",

    ],
    "wav_only": [
        "MeanUnvoicedSegmentLength",
        "StddevUnvoicedSegmentLength",
        "F0semitoneFrom27.5Hz_sma3nz_amean",
        "F0semitoneFrom27.5Hz_sma3nz_stddevNorm",
        "F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2",
        "loudness_sma3_amean",
        "loudness_sma3_stddevNorm",
        "loudness_sma3_pctlrange0-2",
        "loudnessPeaksPerSec",
        "jitterLocal_sma3nz_amean",
        "shimmerLocaldB_sma3nz_amean",
        "HNRdBACF_sma3nz_amean",
        "alphaRatioV_sma3nz_amean",
        "slopeV0-500_sma3nz_amean",
        "slopeV500-1500_sma3nz_amean",
        "equivalentSoundLevel_dBp",
    ],
    "subset_cha": [
        "total_speaking_time_normalized",
        "interviewer_interruptions",
        "word_rate_words_per_sec",
        "mean_unique_words_per_utterance",
        ],
    "subset_wav": [
        "MeanUnvoicedSegmentLength",
        "StddevUnvoicedSegmentLength",
        "loudness_sma3_amean",
        "loudness_sma3_stddevNorm",
        "loudness_sma3_pctlrange0-2",
        "loudnessPeaksPerSec",
        "jitterLocal_sma3nz_amean",
        "shimmerLocaldB_sma3nz_amean",
        "equivalentSoundLevel_dBp",
    ]
    }

In [52]:
# 1. load feature tables
feature_paths = {
    "English": r"D:\masteruwefduyqeahfdqe\ASR-project\full_feature_table_final_pitts.csv",
    "Mandarin": r"D:\masteruwefduyqeahfdqe\ASR-project\full_feature_table_final_chinese.csv",
    "Greek": r"D:\masteruwefduyqeahfdqe\ASR-project\full_feature_table_final_greek.csv",
}

feature_tables = load_feature_tables(feature_paths)
features_df = build_master_feature_table(feature_tables)

In [53]:
# 2. run experiments
predictions_df = run_all_experiments(
    features_df=features_df,
    feature_sets=FEATURE_SETS,
    experiments=EXPERIMENTS,
    get_models_fn=get_models,
    random_state=42,
)


Running feature set: all
  Experiment: train_english_greek_test_mandarin [cross]
Running experiment: train_english_greek_test_mandarin with strategy: cross
Cross-lingual split: 428 training samples from ['English', 'Greek']; 81 test samples from ['Mandarin']
  Experiment: train_mandarin_greek_test_english [cross]
Running experiment: train_mandarin_greek_test_english with strategy: cross
Cross-lingual split: 215 training samples from ['Mandarin', 'Greek']; 159 test samples from ['English']
  Experiment: train_english_mandarin_test_greek [cross]
Running experiment: train_english_mandarin_test_greek with strategy: cross
Cross-lingual split: 569 training samples from ['English', 'Mandarin']; 17 test samples from ['Greek']

Running feature set: cha_only
  Experiment: train_english_greek_test_mandarin [cross]
Running experiment: train_english_greek_test_mandarin with strategy: cross
Cross-lingual split: 428 training samples from ['English', 'Greek']; 81 test samples from ['Mandarin']
  Expe

In [ ]:
# save predictions to CSV
predictions_df.to_csv("model_predictions_final_new_cross.csv", index=False)

In [ ]:
predictions_df

In [ ]:
# check for NAn values in features_df
nan_counts = features_df.isna().sum()
print("NaN counts in features_df:")
print(nan_counts)

# which rows have NaN values in features_df
nan_rows = features_df[features_df.isna().any(axis=1)]
print("Rows with NaN values in features_df:")
nan_rows[["file_id", "language", "label", "std_utterance_word_rate_words_per_sec","std_words_per_utterance"]]